# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the [FAIR^2 Croissant dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the `mlcroissant` library. 

### Dataset Source
The dataset is accessed via its Croissant schema URL and complies with the [MLCommons Croissant format](https://mlcommons.github.io/croissant/).


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This dataset explores mass spectrometry results for proteins extracted from human extracellular vesicles.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. These unique entity identifiers are essential for referencing table, field, and column definitions in the Croissant schema.

In [ ]:
# List all record sets (table-like entities) and their field structure

# Find all record set @ids and describe their fields
record_sets = list(metadata.record_sets)
print(f'Found {len(record_sets)} record sets:')

for recset in record_sets:
    print(f"- RecordSet Name: {recset.name}  @id: {recset.id}")
    print("  Fields:")
    for field in recset.fields:
        print(f"     - {field.name} (id: {field.id}, dtype: {field.data_type})")
    print("")

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. We reference record sets and fields by their Croissant `@id`, which are unique and unambiguous.

In [ ]:
# Use record_set @id to extract data for each table

# Build a list of all record set @ids
record_set_ids = [recset.id for recset in metadata.record_sets]

dataframes = {}
for recset_id in record_set_ids:
    # Load records as list of dicts
    try:
        records = list(dataset.records(record_set=recset_id))
        df = pd.DataFrame(records)
        dataframes[recset_id] = df
        print(f"Loaded {len(df)} rows for RecordSet: {recset_id}")
        print(f"Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Could not load data for record set {recset_id}: {e}")

# Preview the main table if available; pick the first RecordSet as a demo
if record_set_ids:
    main_recset_id = record_set_ids[0]
    print(f"Showing first 5 rows from RecordSet: {main_recset_id}")
    display(dataframes[main_recset_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's analyze numeric fields, filter, normalize, and group data. 
Use field `@id`s for all column accesses (to ensure schema-tracking correctness).

In [ ]:
# For demonstration, try to use a numeric field
# First, pick a RecordSet (e.g., protein summary table) and list its numeric fields

recset = metadata.record_sets[0]  # CHANGE if you know another main table
df = dataframes[recset.id]

# List possible numeric fields
numeric_fields = [field.id for field in recset.fields if field.data_type.lower() in ["float", "integer", "number"] and field.id in df.columns]
print(f"Numeric fields in RecordSet '{recset.name}': {numeric_fields}")

# If we have a numeric field, use it to demonstrate filtering/normalizing
if numeric_fields:
    numeric_field = numeric_fields[0]
    print(f"Using numeric field '{numeric_field}' for EDA.")

    # Set a basic threshold
    threshold = df[numeric_field].quantile(0.75) if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Add normalized column
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()

    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to group by a likely categorical/text field (e.g., 'Protein_Class' or similar)
    possible_group_fields = [field.id for field in recset.fields if field.data_type.lower() in ["text", "string"] and field.id in df.columns]
    group_field = possible_group_fields[0] if possible_group_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} and mean {numeric_field}:")
        display(grouped_df.head())
else:
    print("No numeric fields found for EDA in the first record set.")

## 5. Visualization
Visualize numeric field distributions using matplotlib or pandas' plotting. 
This helps summarize how protein or peptide values are distributed across samples.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

# Plot histogram and boxplot of the numeric field if available
if 'numeric_field' in locals():
    fig, axs = plt.subplots(1,2, figsize=(12,4))
    df[numeric_field].hist(ax=axs[0], bins=30, color='C0', edgecolor='k')
    axs[0].set_title(f"Distribution of {numeric_field}")
    df.boxplot(column=numeric_field, ax=axs[1])
    axs[1].set_title(f"Boxplot of {numeric_field}")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion

In this notebook, we demonstrated how to:
- Load a FAIR^2 Croissant-formatted dataset with `mlcroissant`,
- Enumerate and extract tables (record sets) and inspect their schema,
- Reference all record sets, fields, and columns by their Croissant `@id`s for reproducibility,
- Perform basic exploratory data analysis (EDA) by filtering and normalizing numeric fields, and
- Visualize quantitative data distributions.

You can extend this notebook by exploring additional record sets, visualizing categorical breakdowns, and joining or relating fields across different tables as appropriate for your research question.